# Laboratorio de métodos de búsqueda

---
Nombre: **Alan Jesús Hernández Soto** \
Carrera: **Ingeniería Financiera** \
Materia: **Laboratorio de Aprendizaje Estadístico** \
Tipo de Trabajo: **Laboratorio**

---

Modelos como regresión logística, máquinas de vectores de soporte, y preceptrones multicapa tienen una serie de argumentos que necesitan valores dados antes de poder usarse para ajustarse a un dataset. Estos argumentos se llaman hiperparámetros.

¿Cuál es la diferencia entre un parámetro y un hiperparámetro?

Un parametro es un valor estimado por un modelo a partir de los datos normalmente estimados en el proceso de entrenamiento mientras que un hiperparametro se define manualmente por el individuo antes de iniciar el entrenamiento.

Este laboratorio se centra en métodos de búsqueda para encontrar los mejores hiperparámetros para un modelo dado. Imagina que estás jugando batalla naval:

<center><img src="https://cdn11.bigcommerce.com/s-nehqfptb/images/stencil/670x993/products/40096/100569/battleship-game__73281.1768238533.jpg?c=2"></center>

Una forma de jugar podría ser ir de cuadro en cuadro hasta probar todo el tablero. El problema aquí es que se juega a contrarreloj, debido a que tu oponente puede hundir tus barcos antes de que pruebes todo el tablero.

Si no se tiene la preocupación/limitante de tiempo, probar todas las combinaciones es una opción:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Entra a la documentación de `GridSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).


ENTRADA
- Un modelo o Pipeline, por ejemplo pipeline.
- Un diccionario llamado param_distributions, con valores o distribuciones de donde se tomarán parámetros.
- El número de combinaciones que se desea probar: n_iter.
- Al ejecutar .fit(X_train, y_train), recibe las variables predictoras y la variable objetivo.
No prueba necesariamente todas las combinaciones: toma una muestra aleatoria de ellas.

SALIDA \
Después de .fit(), produce los mismos resultados principales que GridSearchCV:
- best_params_: parámetros de la mejor combinación aleatoria probada.
- best_score_: puntuación promedio de esa combinación.
- best_estimator_: modelo final con esos parámetros.
- cv_results_: resultados de cada combinación seleccionada aleatoriamente.

PARAMETROS IMPORTANTES/ USADOS
- estimator: el modelo a optimizar; en este caso, el pipeline.
- param_grid: diccionario con todas las combinaciones que se quieren probar.
- scoring="roc_auc": criterio usado para decidir qué combinación es mejor.
- cv=cv_10: forma de dividir los datos; aquí son 10 folds.
- n_jobs=-1: usa todos los núcleos disponibles.
- refit=True: vuelve a entrenar el mejor modelo usando todo X_train.
- return_train_score=True: guarda también los resultados de entrenamiento.

Otra forma de "buscar" podría ser tirar al azar y ver qué pasa. Esto sería otra forma de utilizar la fuerza bruta para encontrar información. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

Entra a la documentación de `RandomizedSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros). 

ENTRADA
- Un modelo o Pipeline, por ejemplo pipeline.
- Un diccionario llamado param_distributions, con valores o distribuciones de donde se tomarán parámetros.
- El número de combinaciones que se desea probar: n_iter.
- Al ejecutar .fit(X_train, y_train), recibe las variables predictoras y la variable objetivo.
No prueba necesariamente todas las combinaciones: toma una muestra aleatoria de ellas.

SALIDA
Después de .fit(), produce los mismos resultados principales que GridSearchCV:
- best_params_: parámetros de la mejor combinación aleatoria probada.
- best_score_: puntuación promedio de esa combinación.
- best_estimator_: modelo final con esos parámetros.
- cv_results_: resultados de cada combinación seleccionada aleatoriamente.

PARAMETROS IMPORTANTES/ USADOS
- estimator: el modelo o Pipeline que se quiere optimizar.
- param_distributions: diccionario con opciones o distribuciones para cada parámetro.
- n_iter=15: número de combinaciones aleatorias que se probarán.
- scoring="roc_auc": criterio para elegir el mejor modelo.
- cv=cv_10: usa los 10 folds definidos.
- n_jobs=-1: usa todos los núcleos disponibles.
- random_state=42: hace reproducible la selección aleatoria.
- refit=True: reentrena el mejor modelo con todo X_train.
- return_train_score=True: guarda resultados de entrenamiento además de los de validación.

**Nota:** RandomizedSearchCV prueba puramente al azar. Si encontrara en uno de sus tiros que se atinó a un barco, no cambia el algoritmo de búsqueda para seguir buscando en esa zona. Para un proceso de búsqueda más refinado necesitaríamos un algoritmo que cambie su forma de decidir a partir de información anterior.

Revisa la documentación de Optuna (https://optuna.org/). Instala la librería y escribe una guía de cómo se debe utlizar para un dataset dado.

Optuna sirve para encontrar automáticamente buenos hiperparámetros para un modelo con optimización bayesiana.

Primero se separan los datos en entrenamiento y prueba. El conjunto de prueba se guarda para el final y no se usa durante la búsqueda.

Después se define una función llamada `objective(trial)`. En ella se indican los parámetros que Optuna puede modificar, se crea el modelo con esos valores, se entrena y se devuelve un resultado que indique qué tan bueno fue.

Por ejemplo, para un MLP, Optuna puede elegir el número de capas con `trial.suggest_categorical("n_capas", [2, 3, 4])` y el número de neuronas de cada capa con `suggest_categorical` o `suggest_int`.

Luego se crea un estudio con `optuna.create_study(direction="maximize")`. La dirección debe ser `"maximize"` cuando se busca aumentar una medida como ROC-AUC, accuracy o F1; se usa `"minimize"` para reducir errores como MSE o MAE.

Finalmente se ejecuta `study.optimize(objective, n_trials=30)`. Cada trial representa una configuración distinta de hiperparámetros. Optuna guarda los resultados y usa las pruebas anteriores para proponer configuraciones que parezcan prometedoras.

Al terminar:

- `study.best_params` contiene los mejores hiperparámetros.
- `study.best_value` contiene el mejor resultado obtenido.

Con esos hiperparámetros se entrena un modelo final usando todos los datos de entrenamiento y se evalúa una sola vez con el conjunto de prueba.

Es importante escalar las variables cuando se usa un MLP y no usar el conjunto de prueba dentro de la función `objective`.

### Ejercicio

Utiliza el dataset `College.csv` para intentar predecir si una universidad es privada a partir del resto de los factores.

Utiliza un perceptrón multicapa con hiperparámetros:
- Número de capas intermedias --> ${2, 3, 4}$
- Número de neuronas en capa 1 --> [5, 10]
- Número de neuronas en capa 2 --> [10, 20]
- Número de neuronas en capa 3 (si existe) --> [5, 20]
- Número de neuronas en capa 4 (si existe) --> [5, 10]

Utiliza k-folds cross-validation con $k=10$; optimiza el ROC-AUC de test.

Compara los 3 diferentes métodos de búsqueda descritos en este laboratorio.

In [31]:
import pandas as pd
import numpy as np


df = pd.read_csv('Datasets/College.csv')
df = df.drop(columns=['Unnamed: 0'])
df['Private'] = df['Private'].map({'Yes': 1, 'No': 0})

print(np.sum(df.isnull()))

y = df['Private']
X = df.drop(columns=['Private'])

Private        0
Apps           0
Accept         0
Enroll         0
Top10perc      0
Top25perc      0
F.Undergrad    0
P.Undergrad    0
Outstate       0
Room.Board     0
Books          0
Personal       0
PhD            0
Terminal       0
S.F.Ratio      0
perc.alumni    0
Expend         0
Grad.Rate      0
dtype: int64


c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\fromnumeric.py:81: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


No hay valores nulos

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)


In [33]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(activation="relu", solver="adam", max_iter=2000, early_stopping=True, random_state=67))
])

arquitecturas = []

# 2
for n1 in [5, 10]:
    for n2 in [10, 20]:
        arquitecturas.append((n1, n2))

#3
for n1 in [5, 10]:
    for n2 in [10, 20]:
        for n3 in [5, 20]:
            arquitecturas.append((n1, n2, n3))

#4
for n1 in [5, 10]:
    for n2 in [10, 20]:
        for n3 in [5, 20]:
            for n4 in [5, 10]:
                arquitecturas.append((n1, n2, n3, n4))

cv_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

GridSearchCV

In [40]:
from sklearn.model_selection import GridSearchCV


param_grid = {"mlp__hidden_layer_sizes": arquitecturas}

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring="roc_auc", cv=cv_10, n_jobs=-1, refit=True, return_train_score=True)
grid_search.fit(X_train, y_train)

print("Mejor arquitectura:", grid_search.best_params_)
print("Mejor ROC-AUC promedio CV:", grid_search.best_score_)

Mejor arquitectura: {'mlp__hidden_layer_sizes': (10, 20, 5, 5)}
Mejor ROC-AUC promedio CV: 0.9413205100880931


In [41]:
from sklearn.metrics import roc_auc_score

mjr_m = grid_search.best_estimator_

prob = mjr_m.predict_proba(X_test)[:, 1]

roc_auc_test_grid = roc_auc_score(y_test, prob)

print("ROC-AUC de test:", roc_auc_test_grid)

ROC-AUC de test: 0.8948343280510394


RandomizedSearchCV

In [43]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_grid, n_iter=20, scoring="roc_auc", cv=cv_10, n_jobs=-1, random_state=67, refit=True, return_train_score=True )
random_search.fit(X_train, y_train)

print("Mejor arquitectura:", random_search.best_params_)
print("Mejor ROC-AUC promedio CV:", random_search.best_score_)

Mejor arquitectura: {'mlp__hidden_layer_sizes': (10, 20, 5, 5)}
Mejor ROC-AUC promedio CV: 0.9413205100880931


In [44]:
mjr_m = random_search.best_estimator_

prob = mjr_m.predict_proba(X_test)[:, 1]

roc_auc_test_grid = roc_auc_score(y_test, prob)

print("ROC-AUC de test:", roc_auc_test_grid)

ROC-AUC de test: 0.8948343280510394


Optuna

In [45]:
import optuna
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Capas Ocultas
    n_capas = trial.suggest_categorical( "n_capas", [2, 3, 4] )
    
    # Neuronas en copas obligatorias
    capa_1 = trial.suggest_categorical( "capa_1", [5, 10] )
    
    capa_2 = trial.suggest_categorical( "capa_2", [10, 20] )
    
    # La arquitectura comienza con dos capas
    arquitectura = [capa_1, capa_2]
    
    # 3era capa solo si existe
    if n_capas >= 3: 
        capa_3 = trial.suggest_categorical("capa_3", [5, 20]) 
        arquitectura.append(capa_3)
    
    # 4ta capa solo si existe
    if n_capas == 4: 
        capa_4 = trial.suggest_categorical("capa_4", [5, 10]) 
        arquitectura.append(capa_4)
    
    # Clonar el pipelibe, optuna lo solicita
    modelo = clone(pipeline)
    modelo.set_params( mlp__hidden_layer_sizes=tuple(arquitectura))
    
    scores = cross_val_score(modelo, X_train, y_train, scoring="roc_auc", cv=cv_10, n_jobs=-1)
    return scores.mean()

c:\Users\1\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
study = optuna.create_study( direction="maximize", sampler=optuna.samplers.TPESampler(seed=67) )

study.optimize( objective, n_trials=28, n_jobs=1 )

[I 2026-10-29 15:22:35,422] A new study created in memory with name: no-name-e5c78fb7-278a-4fe3-a2ca-54dc32de0eb4
[I 2026-10-29 15:22:46,492] Trial 0 finished with value: 0.31209985080988917 and parameters: {'n_capas': 3, 'capa_1': 5, 'capa_2': 10, 'capa_3': 5}. Best is trial 0 with value: 0.31209985080988917.
[I 2026-10-29 15:22:50,413] Trial 1 finished with value: 0.8981146987780619 and parameters: {'n_capas': 3, 'capa_1': 10, 'capa_2': 20, 'capa_3': 5}. Best is trial 1 with value: 0.8981146987780619.
[I 2026-10-29 15:22:50,653] Trial 2 finished with value: 0.8260734583688547 and parameters: {'n_capas': 3, 'capa_1': 5, 'capa_2': 10, 'capa_3': 20}. Best is trial 1 with value: 0.8981146987780619.
[I 2026-10-29 15:22:50,884] Trial 3 finished with value: 0.8260734583688547 and parameters: {'n_capas': 3, 'capa_1': 5, 'capa_2': 10, 'capa_3': 20}. Best is trial 1 with value: 0.8981146987780619.
[I 2026-10-29 15:22:51,191] Trial 4 finished with value: 0.9413205100880931 and parameters: {'n_c

In [48]:
print("Mejores:")
print(study.best_params)

print("Mejor ROC-AUC promedio CV:")
print(study.best_value)

Mejores:
{'n_capas': 4, 'capa_1': 10, 'capa_2': 20, 'capa_3': 5, 'capa_4': 5}
Mejor ROC-AUC promedio CV:
0.9413205100880931


In [51]:
mejores = study.best_params

mjr = [mejores["capa_1"], mejores["capa_2"] ]

if mejores["n_capas"] >= 3:
    mjr.append(mejores["capa_3"])

if mejores["n_capas"] == 4:
    mjr.append(mejores["capa_4"])

modelo_test = clone(pipeline)

modelo_test.set_params(mlp__hidden_layer_sizes=tuple(mjr))

modelo_test.fit(X_train, y_train)

prob = modelo_test.predict_proba(X_test)[:, 1]

roc_auc= roc_auc_score(y_test, prob)

print("Mejor arquitectura:", tuple(mjr))
print("ROC-AUC de test:", roc_auc)

Mejor arquitectura: (10, 20, 5, 5)
ROC-AUC de test: 0.8948343280510394
